In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
sdate, edate = '2019-01-06','2019-02-23'

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/gen_details.csv")

In [5]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")
hw_tseries = hw_tseries.drop(['index','points'], axis=1)

In [6]:
def process_group(grp, gen_fpath, hw_tseries, start_date=sdate, end_date=edate):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    gen_locs = gen_fpath + '/' + grp['DUID'] + ".csv"
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]
    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    agg_func = {'TOTALMWh':'sum','TOTALCLEARED':'sum','AGCSTATUS':'min'}
    dfs = dfs.groupby(['DUID', pd.Grouper(freq='1d')]).agg(agg_func)

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        dfs.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("12h"),
        direction='nearest'
    )

    return merged

In [7]:
groups = gen_details.groupby('region',
                            as_index = False)

df = groups.get_group('VIC1')

06/01/2019-12/01/2019 BEFORE
13/01/2019-31/01/2019 DURING 
09/02/2019-16/02/2019 AFTER

In [8]:
# This retrieves only the time surrounding the heatwave
df = process_group(df, gen_fpath, hw_tseries, sdate, edate)
df = df.merge(gen_details[['DUID','fuel_source_primary']], left_on='DUID', right_on='DUID', how='left')

In [9]:
demand = pd.read_csv('/scratch/ng72/ms5578/time_series/state_demand.csv')
demand['time'] = pd.to_datetime(demand['time'])
demand = demand.set_index('time').sort_index()
demand = demand.loc[sdate:edate]
demand = demand[demand["REGIONID"] ==  'VIC1'].drop('REGIONID',axis=1)
demand = demand.resample('d').sum()

In [10]:
gens_in_hw = df.groupby('time').agg({'EHF_flag': 'sum'})

In [17]:
def difference_from_first_week(df, date_col='time', value_col='TOTALMWh', group_cols=['DUID']):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['weekday'] = df[date_col].dt.weekday

    def get_first_week(group):
        first_week_start = group[date_col].min().normalize()
        first_week = group[group[date_col] < first_week_start + pd.Timedelta(days=7)]
        return first_week.set_index('weekday')[[value_col]]

    def compute_diff(group):
        group_data = group.drop(columns=group_cols)
        group_data = group_data.copy()
        group_data[value_col] = group[value_col].values
        group_data[date_col] = group[date_col].values
        group_data['weekday'] = group['weekday'].values

        first_week_vals = get_first_week(group_data)
        group_data = group_data.set_index('weekday')
        group_data['first_week_value'] = group_data.index.map(first_week_vals[value_col].to_dict())
        group_data['diff_from_first_week'] = (group_data[value_col] - group_data['first_week_value'])
        group_data = group_data.reset_index()
        for col in group_cols:
            group_data[col] = group[col].iloc[0]
        return group_data

    result = df.groupby(group_cols, group_keys=False)[df.columns.tolist()].apply(compute_diff)
    return result

diff = difference_from_first_week(df)

In [18]:
agg_func = {'TOTALMWh':'sum','EHF_flag':'max', 'EHF_val':'max','TOTALCLEARED':'sum', 'fuel_source_primary':'first'}
fuel_grp = df.groupby(['DUID', 'time']).aggregate(agg_func).reset_index()
diff = difference_from_first_week(fuel_grp,group_cols=['DUID'])

In [19]:
def plot_agg_group(
    grouped_df, group_col, title='Time Series Plot', y='TOTALMWh',
    highlight=False, highlight_mode='union'
):
    fig = go.Figure()

    for group_name, group in grouped_df.groupby(group_col):
        fig.add_trace(go.Scatter(
            x=group['time'],
            y=group[y],
            mode='lines',
            name=str(group_name)
        ))

    if highlight:
        if highlight_mode == 'union':
            # Highlight where any group has EHF_flag==1 (union)
            highlight_times = (
                grouped_df.groupby('time')['EHF_flag']
                .max()
                .reset_index()
            )
            fig = highLights(
                df=highlight_times,
                fig=fig,
                variable='EHF_flag',
                level=0,
                mode='above',
                fillcolor='rgba(255,0,0,0.1)',
                layer='below'
            )
        elif highlight_mode == 'per_group':
            # Highlight per group
            for group_name, group in grouped_df.groupby(group_col):
                fig = highLights(
                    df=group,
                    fig=fig,
                    variable='EHF_flag',
                    level=0,
                    mode='above',
                    fillcolor='rgba(255,0,0,0.1)',
                    layer='below'
                )
        else:
            raise ValueError("highlight_mode must be 'union' or 'per_group'")

    fig.update_layout(
        title=title,
        xaxis_title='Time',
        yaxis_title='Total MWh',
        template='plotly_white',
        legend_title=group_col
    )

    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(count=1,
                         label="1m",
                         step="month",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )

    return fig

In [20]:
def highLights(df, fig, variable, level, mode, fillcolor, layer):
    """
    Set a specified color as background for given
    levels of a specified variable using a shape.
    
    Keyword arguments:
    ==================
    fig -- plotly figure
    variable -- column name in a pandas dataframe
    level -- int or float
    mode -- set threshold above or below
    fillcolor -- any color type that plotly can handle
    layer -- position of shape in plotly fiugre, like "below"
    
    """
    
    if mode == 'above':
        m = df[variable].gt(level)
    
    if mode == 'below':
        m = df[variable].lt(level)
        
    df1 = df[m].groupby((~m).cumsum())['time'].agg(['first','last'])

    for index, row in df1.iterrows():
        #print(row['first'], row['last'])
        fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0=row['first'],
            y0=0,
            x1=row['last'],
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(100,100,100,0.2)",
            layer=layer
        )
    return(fig)


In [37]:
fig = px.bar(
    diff, 
    x='time', 
    y='diff_from_first_week', 
    color='DUID',
    title='Daily generation difference from pre-heatwave week by DUID',
    labels={'diff_from_first_week': 'MWh difference'
    },
    hover_data=['fuel_source_primary']  # Include it here
)

# Highlight where any group has EHF_flag==1 (union)
hw_times = (
    diff.groupby('time')['EHF_flag']
    .max()
    .reset_index()
)

fig.add_shape(
        type="rect",
        xref="x",
        yref="paper",
        x0='2019-01-13',
        y0=0,
        x1='2019-02-10',
        y1=1,
        line=dict(color="rgba(0,0,0,0)", width=3),
        fillcolor="rgba(0,0,0,0.1)",
        layer='below')

fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0='2019-01-23 12:00:00',
            y0=0,
            x1='2019-01-25 12:00:00',
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(255,0,0,0.2)",
            layer='below')


fig.show()

In [32]:
diff.columns

Index(['weekday', 'time', 'TOTALMWh', 'EHF_flag', 'EHF_val', 'TOTALCLEARED',
       'fuel_source_primary', 'first_week_value', 'diff_from_first_week',
       'DUID'],
      dtype='object')